# Structured User-Course Feature Table (AGENT)

This notebook rebuilds the LMS EDA as a feature-engineering pipeline focused on **one row per `users_course_id`**.

What was retained from the existing project EDA:
- ID normalization must strip thousand separators such as `718,902 -> 718902`.
- Agent accounts are removed via `users.type == 'User::Agent'` rather than the unreliable raw `is_teacher` field.
- Course structure is validated through `lessons` before course-level enrichment is attached to enrollments.

What changed in this AGENT version:
- the pipeline is explicitly organized by entity grain;
- heavy event tables are aggregated block-by-block before merging;
- bridge hypotheses are validated with saved metric tables and figures;
- the final output is a leakage-aware feature table ready for future target attachment.

## Rebuild Path

The headless build script is [`scripts/build_user_course_features_AGENT.py`](../scripts/build_user_course_features_AGENT.py).

Run it from the project root to rebuild all artifacts in `data/AGENT`.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT_ = Path.cwd().resolve().parent
if str(PROJECT_ROOT_) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_))

print(PROJECT_ROOT_)

C:\Repos\Xakaton


In [3]:
from pathlib import Path
import pandas as pd

from scripts.build_user_course_features_AGENT import PROJECT_ROOT, OUT_DIR, build_all_artifacts

# Uncomment to rebuild all AGENT artifacts from raw data.
# artifacts = build_all_artifacts()

final_features = pd.read_csv(OUT_DIR / 'final_user_course_features_AGENT.csv')
final_features.shape

(267206, 228)

## 1. Raw Table Audit

The project audit covers raw tables in `data/raw`, keeping raw files untouched and saving all derived outputs to `data/AGENT`.

### Table overview

| table_name | row_count | column_count | file_size_mb | estimated_memory_mb | table_role |
| --- | --- | --- | --- | --- | --- |
| user_answers | 15176182 | 15 | 2141.53 | 8518.93 | event |
| wk_users_courses_actions | 12909207 | 8 | 1025.97 | 4891.84 | event |
| user_lessons | 3070664 | 12 | 227 | 874.56 | event |
| user_activity_histories | 3031137 | 4 | 151.71 | 624.96 | event |
| wk_media_view_sessions | 852358 | 8 | 56.32 | 288.11 | event |
| user_access_histories | 667124 | 5 | 60.12 | 165.16 | event |
| user_trainings | 427628 | 13 | 58.88 | 220.9 | entity_or_small_event |
| users_courses | 290835 | 14 | 33.01 | 139.52 | entity_or_small_event |
| user_award_badges | 252843 | 4 | 9.95 | 37.55 | entity_or_small_event |
| users | 95395 | 22 | 20.41 | 87.47 | entity_or_small_event |
| lesson_tasks | 29544 | 6 | 1.12 | 4.28 | entity_or_small_event |
| groups | 13076 | 13 | 1.36 | 6.75 | entity_or_small_event |
| homework_items | 5901 | 6 | 0.2 | 1.19 | entity_or_small_event |
| lessons | 3369 | 13 | 0.19 | 0.47 | entity_or_small_event |
| homeworks | 1226 | 5 | 0.03 | 0.17 | entity_or_small_event |
| trainings | 410 | 9 | 0.05 | 0.15 | entity_or_small_event |
| award_badges | 6 | 8 | 0 | 0 | entity_or_small_event |

![Raw table rows](../data/AGENT/figures/raw_table_rows_AGENT.png)

### Candidate key diagnostics

| table_name | candidate_key | row_count | null_key_rows | duplicate_rows_on_key | unique_key_rows | is_unique_key |
| --- | --- | --- | --- | --- | --- | --- |
| user_access_histories | users_course_id, access_started_at, access_expired_at | 667124 | 0 | 355381 | 311743 | 0 |
| groups | id | 13076 | 0 | 0 | 13076 | 1 |
| lesson_tasks | lesson_id, task_id, position | 29544 | 0 | 0 | 29544 | 1 |
| lessons | id | 3369 | 0 | 0 | 3369 | 1 |
| trainings | id | 410 | 0 | 0 | 410 | 1 |
| user_award_badges | user_id, award_badge_id, created_at | 252843 | 0 | 0 | 252843 | 1 |
| user_lessons | users_course_id, lesson_id | 3070664 | 0 | 0 | 3070664 | 1 |
| users | id | 95395 | 0 | 0 | 95395 | 1 |
| users_courses | id | 290835 | 0 | 0 | 290835 | 1 |
| users_courses | user_id, course_id | 290835 | 0 | 0 | 290835 | 1 |

## 2. Entity Map

Every table is assigned a native grain, a target grain, and a merge strategy before any raw merge is attempted.

| table_name | represents | native_grain | candidate_key | target_grain | merge_strategy | use_in_pipeline |
| --- | --- | --- | --- | --- | --- | --- |
| users | Platform user profile | one row per user | id | user | Merge directly on user_id after agent filtering | yes |
| users_courses | Course enrollment | one row per user-course enrollment | id | user-course | Core entity; keep one row per users_course_id | yes |
| lessons | Course lesson structure | one row per lesson | id | course | Aggregate to course_id before merging | yes |
| lesson_tasks | Task-to-lesson links | one row per lesson-task link | (lesson_id, task_id, position) | course | Aggregate via lesson_id -> course_id | yes |
| groups | Webinar/group schedule | one row per group | id | course | Aggregate via lesson_id -> course_id | yes |
| trainings | Training metadata | one row per training template | id | course | Aggregate via lesson_id -> course_id | yes |
| user_lessons | User progress on lesson | one row per user-lesson state | (users_course_id, lesson_id) | user-course | Aggregate on users_course_id | yes |
| user_trainings | User training attempts | one row per user-training attempt | (user_id, training_id, started_at) | user-course | Resolve course via trainings -> lessons -> course_id, then map by (user_id, course_id) | yes |
| user_answers | Task answer events | one row per submitted answer | event-like, no clean natural key in export | user-course | Resolve course by resource_type/resource_id before aggregation | yes |
| wk_users_courses_actions | User-course action log | one row per action | event-like | user-course | Aggregate directly on users_course_id | yes |
| wk_media_view_sessions | Media sessions | one row per media session | event-like | user-course | Resolve course via Lesson/Group resources, then map by (viewer_id, course_id) | yes |
| user_access_histories | Access window history | one row per access interval event | (users_course_id, access_started_at, access_expired_at) | user-course | Aggregate directly on users_course_id | yes |
| user_award_badges | Assigned badge event | one row per badge assignment | (user_id, award_badge_id, created_at) | user | Aggregate on user_id; use as user-level enrichment | yes |
| award_badges | Badge dictionary | one row per badge type | id | dictionary | Reference only; not used as standalone feature block | reference_only |
| user_activity_histories | Alternative activity log | one row per activity event | event-like | excluded | Audited but excluded from the main feature pipeline | excluded |

## 3. Core Entity: `users_course_id`

After removing agent accounts, the core enrollment table contains **267,206 rows** and remains unique both on `users_course_id` and on `(user_id, course_id)`.

| metric | value |
| --- | --- |
| users_course_id_unique | 1 |
| user_course_pair_unique | 1 |
| rows_after_agent_filter | 267206 |

The final ML table keeps enrollment and access anchors but excludes completion-derived leakage fields.

## 4. Course Structure Block

Course-level features are built from `lessons`, `lesson_tasks`, `groups`, and `trainings`, all aggregated to `course_id` first.

### Lessons-based reconciliation against enrollment-level course fields

| metric | mean_abs_delta | median_abs_delta |
| --- | --- | --- |
| max_points_delta | 10.709 | 0 |
| lesson_count_delta | 6.747 | 3 |
| task_count_delta | 10.772 | 0 |

![Course points validation](../data/AGENT/figures/course_points_validation_AGENT.png)

### Lesson ordering caveat

`lesson_number` coverage is partial, so progression-by-order is treated as optional and evidence-based rather than universal.

| stat | value |
| --- | --- |
| count | 137 |
| mean | 0.223 |
| std | 0.385 |
| min | 0 |
| 25% | 0 |
| 50% | 0 |
| 75% | 0.4 |
| max | 1 |

## 5. User-Level Enrichment

User features are merged only at the user grain after agent filtering. The block includes sign-in activity, grade and geography identifiers, subscription status, account creation timestamp, and lightweight badge-event signals from `user_award_badges`.

## 6. User-Lesson Block

`user_lessons` is aggregated directly on `users_course_id`. The block captures visit counts, solved lessons, solved task totals, lesson points, unique lessons touched, group interactions, and partial progression features derived from `lesson_number` where that ordering is available.

## 7. User-Training Block

Training behavior is never merged naively. The resolution path is:
- `user_trainings.training_id -> trainings.id`
- `trainings.lesson_id -> lessons.id`
- `lessons.course_id`
- `(user_id, course_id) -> users_courses.users_course_id`

| block_name | source_rows | course_resolved_rows | enrollment_resolved_rows | course_resolved_share | enrollment_resolved_share |
| --- | --- | --- | --- | --- | --- |
| user_trainings | 421814 | 410688 | 410688 | 0.9736 | 0.9736 |

## 8. User-Answers Block

`task_id` is **not** used as a direct course bridge because the same task appears in multiple courses. Instead, answers are resolved through `resource_type/resource_id` with separate paths for `Lesson`, `Training`, and observed `Homework` resources.

| resource_type | rows | course_resolved_rows | course_resolved_share |
| --- | --- | --- | --- |
| Homework | 399598 | 399595 | 1 |
| Lesson | 9666796 | 9657128 | 0.999 |
| Training | 4889405 | 4857186 | 0.9934 |

The aggregated block contains answer intensity, solved/unsolved/skipped counts, attempts, points, partial-answer signals, async checking counts, distinct answered tasks, type shares, and answer timing anchors.

## 9. Course Actions Block

`wk_users_courses_actions` is the main time-aware behavior source. The block aggregates total actions, action mix by type, active days, early-window activity, peak-day timing, inactivity gaps, and recency signals.

![Action intensity profile](../data/AGENT/figures/action_intensity_profile_AGENT.png)

![Action volume distribution](../data/AGENT/figures/actions_total_AGENT.png)

## 10. Media and Access Blocks

Media sessions are resolved to course via either `Lesson` resources or `Group -> lesson -> course` resources. Access windows are aggregated directly by `users_course_id`.

### Media bridge evidence

| resource_type | rows | course_resolved_rows | course_resolved_share |
| --- | --- | --- | --- |
| Group | 220344 | 220344 | 1 |
| Lesson | 612697 | 612697 | 1 |

The access-history block adds access-period counts, access duration totals, access gaps, reopen flags, and activator-class counts.

## 11. Merge Validation

Every block is merged one by one into the enrollment base, with row preservation, duplicate checks, and coverage tracking.

| block_name | rows_before_merge | rows_after_merge | users_course_id_duplicate_rows | matched_rows | unmatched_rows | coverage_share | new_columns_count | mean_new_column_null_share | max_new_column_null_share |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| course_features | 267206 | 267206 | 0 | 267206 | 0 | 1 | 42 | 0.1778 | 0.737 |
| user_features | 267206 | 267206 | 0 | 267206 | 0 | 1 | 23 | 0.1491 | 0.9439 |
| user_lessons | 267206 | 267206 | 0 | 208629 | 58577 | 0.7808 | 15 | 0.3988 | 0.902 |
| user_trainings | 267206 | 267206 | 0 | 161517 | 105689 | 0.6045 | 23 | 0.4486 | 1 |
| user_answers | 267206 | 267206 | 0 | 170022 | 97184 | 0.6363 | 25 | 0.4127 | 0.9635 |
| course_actions | 267206 | 267206 | 0 | 208519 | 58687 | 0.7804 | 33 | 0.3101 | 1 |
| media_sessions | 267206 | 267206 | 0 | 95249 | 171957 | 0.3565 | 13 | 0.6435 | 0.6435 |
| access_history | 267206 | 267206 | 0 | 267114 | 92 | 0.9997 | 13 | 0.136 | 0.8824 |

![Merge coverage](../data/AGENT/figures/merge_coverage_AGENT.png)

![Bridge coverage](../data/AGENT/figures/bridge_coverage_AGENT.png)

## 12. Feature Engineering

Feature engineering is performed inside each block before the final assembly and then extended with cross-block features. Main families of engineered features:
- structural context: lesson counts, tasks, videos, trainings, webinar structure;
- user context: sign-ins, subscription, grade, geography, badge-event history;
- progression: visited lesson ratios, solved lesson ratios, training lesson reach, fraction of ordered course reached where available;
- intensity: action counts, answer density, media sessions per lesson, points ratios;
- timing: first and last observed action/answer/training/media timestamps relative to course start;
- slowdown and decay: early-window activity, peak activity day, gap statistics between active days, recency gap to access end.

Completion-derived fields are intentionally excluded from the final table even though they remain available in the base export for future target-building work.

## 13. Feature Quality Review

### Feature block summary

| block_name | feature_count |
| --- | --- |
| base | 20 |
| course_features | 42 |
| user_features | 23 |
| user_lessons | 15 |
| user_trainings | 23 |
| user_answers | 25 |
| course_actions | 33 |
| media_sessions | 13 |
| access_history | 13 |
| final_total | 227 |

### Top missing features

| feature_name | missing_share |
| --- | --- |
| action_unique_lessons_touched | 0.9953 |
| first_answer_lesson_number | 0.9635 |
| furthest_answer_lesson_number | 0.9635 |
| wk_gender | 0.9439 |
| first_solved_lesson_number | 0.902 |
| furthest_lesson_number_reached | 0.8895 |
| progression_span_in_lessons | 0.8895 |
| first_lesson_number_seen | 0.8895 |
| fraction_of_course_reached_by_lesson_number | 0.8895 |
| actual_vs_planned_duration_ratio_mean | 0.737 |
| actual_webinar_duration_max | 0.737 |
| actual_webinar_duration_mean | 0.737 |

![Top missing features](../data/AGENT/figures/final_missingness_top20_AGENT.png)

![Visited lessons ratio](../data/AGENT/figures/visited_lessons_ratio_AGENT.png)

![First activity delay](../data/AGENT/figures/first_activity_delay_AGENT.png)

![Last activity recency](../data/AGENT/figures/last_activity_recency_AGENT.png)

### Dropped exact-constant features

| dropped_constant_feature |
| --- |
| regular_trainings_count |
| olympiad_trainings_count |

## 14. Final Output

The final AGENT feature table has **267,206 rows** and **228 columns** after dropping exact constants and obvious leakage fields.

Saved outputs:
- `data/AGENT/users_courses_base_AGENT.csv`
- `data/AGENT/course_features_AGENT.csv`
- `data/AGENT/user_features_AGENT.csv`
- `data/AGENT/user_lessons_agg_AGENT.csv`
- `data/AGENT/user_trainings_agg_AGENT.csv`
- `data/AGENT/user_answers_agg_AGENT.csv`
- `data/AGENT/course_actions_agg_AGENT.csv`
- `data/AGENT/media_sessions_agg_AGENT.csv`
- `data/AGENT/access_history_agg_AGENT.csv`
- `data/AGENT/final_user_course_features_AGENT.csv`

A small preview of the final table is shown below.

| user_id | course_id | state | created_at | access_finished_at | wk_points | wk_max_points | wk_max_viewable_lessons | wk_max_task_count | wk_officially_started_at | users_course_id | course_started_flag | access_is_active_flag | user_course_points_ratio | course_anchor_at | days_created_to_anchor | days_anchor_to_access_end | max_points_missing_flag | max_lessons_missing_flag | max_tasks_missing_flag | lessons_count | tasks_total_from_lessons | max_points_total_from_lessons | total_video_duration | average_video_duration | median_video_duration | share_of_lessons_with_tasks | share_of_lessons_with_conspect | share_of_lessons_with_survival_training | share_of_lessons_with_scratch | share_of_lessons_with_attendance_tracking | video_available_share | lesson_number_available_share | course_min_lesson_number | course_max_lesson_number | lesson_task_links_total | unique_tasks_total | required_tasks_total | required_task_share | trainings_total | published_trainings_count | training_time_limit_mean | training_time_limit_max | trainings_difficulty_mean | training_task_templates_total | training_task_templates_mean | lessons_with_trainings_count | groups_count_per_course | planned_webinar_duration_total | planned_webinar_duration_mean | video_available_share_groups | finished_webinar_count | started_webinar_count | actual_webinar_duration_mean | actual_webinar_duration_max | actual_vs_planned_duration_ratio_mean | webinar_notification_lead_hours_mean | tasks_per_lesson_mean | groups_per_lesson_mean | trainings_per_lesson_mean | lessons_with_trainings_share | published_trainings_share | user_created_at | sign_in_count | grade_id | subscribed_flag | school_id | municipal_id | region_id | wk_gender | has_school_id_flag | has_municipal_id_flag | has_region_id_flag | gender_known_flag | badge_events_total | unique_badges_total | first_badge_at | last_badge_at | has_badge_1_flag | has_badge_2_flag | has_badge_3_flag | has_badge_4_flag | has_badge_5_flag | has_badge_6_flag | badge_activity_span_days | user_lessons_rows | visited_video_lessons_count | viewed_video_completion_count | visited_translation_lessons_count | solved_lessons_count | solved_tasks_total | points_total_over_lessons | unique_group_rows_count | first_lesson_number_seen | first_solved_lesson_number | furthest_lesson_number_reached | points_mean_over_lessons | unique_lessons_touched | unique_groups_visited | progression_span_in_lessons | trainings_started_count | trainings_finished_count | trainings_checked_count | lesson_trainings_count | training_attempts_total | training_attempts_mean | submitted_answers_total_in_trainings | solved_tasks_total_in_trainings | earned_points_total_in_trainings | mark_mean | mark_max | mark_min | count_mark_ge_4 | first_training_started_at | last_training_finished_at | unique_training_lessons_count | unique_training_templates_count | passed_training_ratio | training_activity_span_days | answers_count | solved_answers_count | unsolved_answers_count | skipped_count | attempts_total | points_total | partial_answer_count | async_checked_count | lesson_answer_count | training_answer_count | homework_answer_count | first_attempt_success_count | first_answer_at | first_answer_lesson_number | last_answer_at | furthest_answer_lesson_number | max_attempt_pressure | attempts_mean | points_mean | answered_tasks_count_unique | answer_active_days_count | answer_activity_span_days | success_ratio_over_answers | average_points_per_answer | proportion_of_tasks_solved_on_first_attempt | actions_total | start_training_count | user_answer_action_count | visit_translation_count | visit_video_count | visit_preparation_material_count | scratch_playground_visit_count | first_action_at | last_action_at | action_unique_lessons_touched | active_days_count | actions_first_1d | actions_first_3d | actions_first_7d | actions_first_14d | actions_first_30d | active_days_first_7d | active_days_first_14d | active_days_first_30d | active_days_after_30d | last_action_day_from_start | mean_gap_between_active_days | max_gap_between_active_days | longest_inactivity_streak_days | days_from_course_start_to_peak_activity | peak_actions_in_day | action_span_days | mean_actions_per_active_day | unique_action_types_count | preparation_material_engagement_flag | scratch_engagement_flag | media_sessions_count | lesson_media_sessions_count | group_media_sessions_count | viewed_segments_total | viewed_segments_mean | viewed_fraction_mean | viewed_fraction_max | fully_watched_sessions_count | first_media_session_at | last_media_session_at | unique_resources_viewed | media_active_days_count | media_activity_span_days | access_periods_count | first_access_started_at | last_access_expired_at | total_access_days | current_access_window_length | premium_access_event_count | revoke_access_event_count | change_duration_event_count | standard_access_event_count | month_premium_access_event_count | mean_access_gap_days | max_access_gap_days | access_reopen_flag | account_age_at_enrollment_days | account_age_at_course_start_days | visited_lessons_ratio_vs_course_total | solved_lessons_ratio_vs_course_total | solved_tasks_ratio_vs_course_total | training_progress_ratio_over_course_structure | answer_density_per_lesson | media_sessions_per_lesson | actions_per_lesson | fraction_of_course_reached_by_lesson_number | days_from_course_start_to_first_action | days_from_course_start_to_first_answer | days_from_course_start_to_first_training | days_from_course_start_to_first_media_session | days_from_course_start_to_last_action | days_from_course_start_to_last_answer | days_from_course_start_to_last_training | days_from_course_start_to_last_media_session | first_observed_activity_at | last_observed_activity_at | days_from_course_start_to_first_observed_activity | days_from_last_observed_activity_to_access_end | overall_observed_activity_span_days | early_to_late_actions_ratio_14d | frontloaded_actions_share_7d | frontloaded_active_days_share_14d | days_from_peak_activity_to_last_action |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 665740 | 754 | active | 2025-02-07 11:33:00 | 2025-08-31 |  | 71 | 20 | 71 |  | 449032 | 0 | 1 |  | 2025-02-07 11:33:00 | 0 | 204.5188 | 0 | 0 | 0 | 21 | 71 | 71 | 20 | 1 | 1 | 0.9524 | 0 | 0 | 0 | 0 | 0.9524 | 0 |  |  | 71 | 71 | 71 | 1 |  |  |  |  |  |  |  |  | 21 | 65 | 3.0952 | 0.9524 | 21 | 21 |  |  |  |  | 3.381 | 1 |  |  |  | 2025-01-31 14:16:00 | 21 | 3012 | 0 | 93422 | 93421 | 93420 |  | 1 | 1 | 1 | 0 |  |  |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 |  | 0 |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  | 0 | 0 |  | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 |  |  | 0 |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 1 | 2025-02-07 | 2025-08-31 | 206 | 206 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 6.8868 | 6.8868 |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |
| 665740 | 170000688 | active | 2025-02-11 11:59:00 | 2025-06-30 | 0 | 66 | 29 | 64 |  | 449033 | 0 | 1 | 0 | 2025-02-11 11:59:00 | 0 | 138.5007 | 0 | 0 | 0 | 9 | 16 | 17 | 68 | 7.5556 | 8 | 0.8889 | 0.8889 | 0 | 0 | 0 | 1 | 0 |  |  | 16 | 16 | 16 | 1 |  |  |  |  |  |  |  |  | 9 | 68 | 7.5556 | 0.8889 | 9 | 9 |  |  |  |  | 1.7778 | 1 |  |  |  | 2025-01-31 14:16:00 | 21 | 3012 | 0 | 93422 | 93421 | 93420 |  | 1 | 1 | 1 | 0 |  |  |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 |  | 0 |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  | 0 | 0 |  | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 |  |  | 0 |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 1 | 2025-02-11 | 2025-06-30 | 140 | 140 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 10.9049 | 10.9049 |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |
| 665741 | 754 | active | 2025-02-17 06:45:00 | 2025-06-30 |  | 71 | 20 | 71 |  | 449034 | 0 | 1 |  | 2025-02-17 06:45:00 | 0 | 132.7188 | 0 | 0 | 0 | 21 | 71 | 71 | 20 | 1 | 1 | 0.9524 | 0 | 0 | 0 | 0 | 0.9524 | 0 |  |  | 71 | 71 | 71 | 1 |  |  |  |  |  |  |  |  | 21 | 65 | 3.0952 | 0.9524 | 21 | 21 |  |  |  |  | 3.381 | 1 |  |  |  | 2025-02-17 06:45:00 | 2 | 3012 | 0 | 144476 | 144475 | 144474 | 1 | 1 | 1 | 1 | 1 |  |  |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 |  | 0 |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  | 0 | 0 |  | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 |  |  | 0 |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 1 | 2025-02-17 | 2025-06-30 | 134 | 134 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |
| 665749 | 50000592 | active | 2025-02-17 08:06:00 | 2025-06-30 |  | 329 | 31 | 328 |  | 449049 | 0 | 1 |  | 2025-02-17 08:06:00 | 0 | 132.6625 | 0 | 0 | 0 | 33 | 328 | 329 | 652 | 21.0323 | 21 | 0.9394 | 0.0303 | 0 | 0 | 0 | 0.9394 | 0 |  |  | 308 | 308 | 308 | 1 | 2 | 2 | 75 | 90 | 3 | 20 | 10 | 2 | 33 | 772 | 23.3939 | 0.9394 | 33 | 33 |  |  |  |  | 9.9394 | 1 | 0.0606 | 0.0606 | 1 | 2025-02-17 08:06:00 | 3 | 3012 | 0 | 144476 | 144475 | 144474 | 1 | 1 | 1 | 1 | 1 |  |  |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 |  | 0 |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  | 0 | 0 |  | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 |  |  | 0 |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 1 | 2025-02-17 | 2025-06-30 | 134 | 134 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |
| 665750 | 754 | active | 2025-02-17 08:16:00 | 2025-06-30 |  | 71 | 20 | 71 |  | 449050 | 0 | 1 |  | 2025-02-17 08:16:00 | 0 | 132.6556 | 0 | 0 | 0 | 21 | 71 | 71 | 20 | 1 | 1 | 0.9524 | 0 | 0 | 0 | 0 | 0.9524 | 0 |  |  | 71 | 71 | 71 | 1 |  |  |  |  |  |  |  |  | 21 | 65 | 3.0952 | 0.9524 | 21 | 21 |  |  |  |  | 3.381 | 1 |  |  |  | 2025-02-17 08:16:00 | 3 | 3012 | 0 | 144476 | 144475 | 144474 | 1 | 1 | 1 | 1 | 1 |  |  |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 |  | 0 |  |  |  |  |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  | 0 | 0 |  | 0 | 0 |  |  |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 |  |  |  |  |  |  |  |  |  |  | 0 | 0 | 0 |  |  | 0 |  | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  | 0 | 0 | 1 | 2025-02-17 | 2025-06-30 | 134 | 134 | 1 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |  | 0 | 0 | 0 | 0 | 0 | 0 |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |  |

In [4]:
final_features = pd.read_csv(OUT_DIR / 'final_user_course_features_AGENT.csv')
final_features[['users_course_id', 'user_id', 'course_id']].head()

,users_course_id,user_id,course_id
0,449032,665740,754
1,449033,665740,170000688
2,449034,665741,754
3,449049,665749,50000592
4,449050,665750,754
